In [2]:
#ROW_NUMBER() or RANK() use karke top customers nikalna.

In [3]:
#Dono funnctions rows ko  number dete hai based on kisi order ke, lekin ties(barabar values) handle karne ka tarika alag hai.

In [4]:
#1.ROW_NUMBER()- har row ko unique number dete hai(1,2,3,4...), chahe values barabar hi kyu na ho.

In [5]:
#2.RANK() - barabar values ko same rank deta hai, lekin agla rank skip ho jata hai(1,2,3,4..).

In [6]:
#01:ROW_NUMBER() - top customers by total spend, ek unique ranking:

In [10]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('../data/db/ecommerce.db')

q1 = pd.read_sql("""
    SELECT "Customer ID", Country, total_spend,
            ROW_NUMBER() OVER (ORDER BY total_spend DESC) as rank_num
    FROM (
        SELECT c."Customer ID", c.Country, SUM(o.OrderValue) as total_spend
        FROM customers c
        INNER JOIN orders o ON c."Customer ID" = o."Customer ID"
        GROUP BY c."Customer ID", c.Country
    )
    ORDER BY rank_num
    LIMIT 10
""",conn)
print(q1)                                                                                                                                                                                                                                                                                                                                                                                                 

   Customer ID         Country  total_spend  rank_num
0      18102.0  United Kingdom    598215.22         1
1      14646.0     Netherlands    523342.07         2
2      14156.0            EIRE    296564.69         3
3      14911.0            EIRE    270248.53         4
4      17450.0  United Kingdom    233579.39         5
5      13694.0  United Kingdom    190825.52         6
6      17511.0  United Kingdom    171885.98         7
7      12415.0       Australia    143269.29         8
8      16684.0  United Kingdom    141502.25         9
9      15061.0  United Kingdom    136391.48        10


In [11]:
#02:RANK()- same query, dekho ties kaise handle hoti hai.

In [13]:
q2 = pd.read_sql("""
    SELECT "Customer ID" Country, total_spend,
            RANK() OVER (ORDER BY total_spend DESC) as rank_val
    FROM(
        SELECT c."Customer ID", c.Country, SUM(o.OrderValue) as total_spend
        FROM customers c
        INNER JOIN orders o ON c."Customer ID" = o."Customer ID"
        GROUP BY c."Customer ID",c.Country
    )
    ORDER BY rank_val
    LIMIT 10
""",conn)
print(q2)

   Country  total_spend  rank_val
0  18102.0    598215.22         1
1  14646.0    523342.07         2
2  14156.0    296564.69         3
3  14911.0    270248.53         4
4  17450.0    233579.39         5
5  13694.0    190825.52         6
6  17511.0    171885.98         7
7  12415.0    143269.29         8
8  16684.0    141502.25         9
9  15061.0    136391.48        10


In [14]:
#03:PARTITION BY ke saath - har Country ke anadar top 3 customers nikalna(bahut common real-world pattern):

In [16]:
q3 = pd.read_sql("""
    SELECT *
    FROM (
        SELECT c."Customer ID", c.Country, SUM(o.OrderValue) as total_spend,
                ROW_NUMBER() OVER (PARTITION BY c.Country ORDER BY SUM(o.OrderValue) DESC) as country_rank
        FROM customers c
        INNER JOIN orders o ON c."Customer ID" = o."Customer ID"
        GROUP BY c."Customer ID", c.Country
    )
    WHERE country_rank <= 3
    ORDER BY Country, country_rank
""",conn)
print(q3)


    Customer ID         Country  total_spend  country_rank
0       12415.0       Australia    143269.29             1
1       12431.0       Australia     10719.41             2
2       12422.0       Australia      4119.35             3
3       12429.0         Austria      7435.51             1
4       12370.0         Austria      4320.31             2
..          ...             ...          ...           ...
93      13694.0  United Kingdom    190825.52             3
94      16320.0     Unspecified      4428.85             1
95      14265.0     Unspecified      1373.35             2
96      12363.0     Unspecified       552.00             3
97      18140.0     West Indies       536.41             1

[98 rows x 4 columns]


In [17]:
#yeh pattern bahut important hai - har group me set top N nikalna (jaise "har country ka top 3 customer") sirf window functions se hi clean tarike se hota hai; normal GROUP BY se yeh possible nahi.

In [18]:
#04:Ties dikhane ke liye ek chhota synthetic example(taaki farak clearly dikhe): 

In [19]:
sample = pd.DataFrame({
    'name': ['A', 'B', 'C', 'D'],
    'score': [100, 100, 90, 80]
})
sample.to_sql('sample_scores', conn, if_exists='replace', index=False)

q4 = pd.read_sql("""
    SELECT name, score,
           ROW_NUMBER() OVER (ORDER BY score DESC) as row_num,
           RANK() OVER (ORDER BY score DESC) as rank_val
    FROM sample_scores
""", conn)
print(q4)

  name  score  row_num  rank_val
0    A    100        1         1
1    B    100        2         1
2    C     90        3         3
3    D     80        4         4


In [20]:
#yaha A or B dono ke score 100 hai- ROE_NUMBER() unhe 1,2 dega(arbitrary order), lekin RANK() dono ko 1 dega or agla C ko 3 milega(2 skip ho jaiga).

In [21]:
#practice questions.

In [22]:
#1.Har country mein sabse zyada order value wala sirf 1 customer nikaalo (country_rank = 1 filter karke).

In [23]:
#

In [24]:
#2.Overall top 20 customers RANK() se nikaalo aur dekho kahin koi tie to nahi (agar ho to note karo).

In [25]:
#

In [26]:
conn.close()